# Focused CFL dataset run

A thin front end to the governed pure, bare, self-bound CFL workflow. It stores complete validated tables while suppressing per-case technical PNGs, then creates exactly five combined saved-data figures.

The experimental `dataset_40` profile uses one 40-pressure stellar sequence at the final STRICT ODE tolerances. It is not a full STRICT convergence certificate. First run with `EXECUTE_REVIEWED_PLAN=False`, review the exact Cartesian work, then change only that flag to `True`.

In [ ]:
from eos_generation.notebook import NotebookSettings, get_notebook_session

notebook_session = get_notebook_session()

## Settings

Edit only the next cell. Scalars or lists form a Cartesian product. All center, width, and ramp coordinates are total energy density in MeV fm$^{-3}$. The CFL anchor is always the undeformed zero-pressure self-bound surface. Start with a small grid and review the reported physical-case and stellar-target counts before execution.

In [ ]:
# Edit only this cell. Scalars or lists define the deformation grid.
AMPLITUDES = [-0.10, 0.0, 0.10]
CENTER = 450.0
WIDTH = 300.0
RAMP_WIDTH = 150.0

CALCULATION = "stellar"
FIXED_MASSES = [1.4]
PRECISION = "dataset_40"
DIAGNOSTICS = "off"

# First pass: preview only. Change only this flag after reviewing the plan.
EXECUTE_REVIEWED_PLAN = False

In [ ]:
if CALCULATION != "stellar" or PRECISION != "dataset_40" or DIAGNOSTICS != "off":
    raise ValueError("This notebook requires stellar calculation, CFL dataset_40 precision, and diagnostics off.")
if not isinstance(EXECUTE_REVIEWED_PLAN, bool):
    raise TypeError("EXECUTE_REVIEWED_PLAN must be exactly False or True.")

settings = NotebookSettings.from_values(
    matter_model="cfl",
    epsilon_match="surface",
    amplitudes=AMPLITUDES,
    center=CENTER,
    width=WIDTH,
    ramp_width=RAMP_WIDTH,
    calculation=CALCULATION,
    fixed_masses=FIXED_MASSES,
    precision=PRECISION,
    diagnostics=DIAGNOSTICS,
)

In [ ]:
import hashlib

presentation_sources = {
    name: hashlib.sha256((notebook_session.repository_root / "notebooks" / name).read_bytes()).hexdigest()
    for name in ("eos_catalogue.py", "build_dataset_plots.py")
}
if not EXECUTE_REVIEWED_PLAN:
    reviewed_presentation_sources = presentation_sources
elif globals().get("reviewed_presentation_sources") != presentation_sources:
    raise RuntimeError("Presentation source changed or was not previewed; Run All with EXECUTE_REVIEWED_PLAN=False first.")

notebook_run = notebook_session.prepare(
    settings, record_preview=not EXECUTE_REVIEWED_PLAN
)
print(notebook_run.summary_text())

In [ ]:
experiment_result = notebook_session.execute(
    notebook_run, current_settings=settings, execute=EXECUTE_REVIEWED_PLAN,
)
if experiment_result is None:
    print("EXECUTE_REVIEWED_PLAN=False: no calculations or writes.")
else:
    import json
    import subprocess
    import sys
    from pathlib import Path
    from IPython.display import Markdown, display

    print(experiment_result.summary_text())
    if reviewed_presentation_sources != {
        name: hashlib.sha256((notebook_session.repository_root / "notebooks" / name).read_bytes()).hexdigest()
        for name in reviewed_presentation_sources
    }:
        raise RuntimeError("Scientific run complete but presentation source changed. Preserve packets; recover reporting separately.")
    root = notebook_session.repository_root
    eos_data = notebook_run.planning_root / "EOS_DATA"
    plots = notebook_run.planning_root / "plots"
    try:
        result = subprocess.run(
            [sys.executable, str(root / "notebooks/eos_catalogue.py"),
             "--repository-root", str(root), "--experiment", str(notebook_run.output_root),
             "--destination", str(eos_data)],
            cwd=root, check=True, capture_output=True, text=True,
        )
        catalogue_result = json.loads(result.stdout)
        result = subprocess.run(
            [sys.executable, str(root / "notebooks/build_dataset_plots.py"),
             "--repository-root", str(root), "--experiment", str(notebook_run.output_root),
             "--destination", str(plots), "--eos-data", str(eos_data)],
            cwd=root, check=True, capture_output=True, text=True,
        )
        plot_result = json.loads(result.stdout)
        print(f"Five combined plot families; {plot_result['unique_eos_count']} physical EoSs; plotting solver calls: 0.")
    except (subprocess.CalledProcessError, json.JSONDecodeError) as error:
        print("Scientific calculation is complete. Do not rerun it for a presentation failure.")
        print(getattr(error, "stderr", "") or str(error))
        raise
    locations = {
        "Five combined plots": plots,
        "Labelled primary tables": eos_data,
        "Friendly EoS catalogue": eos_data / "eos_catalogue.csv",
        "Canonical case mapping": eos_data / "case_aliases.csv",
        "Authoritative experiment": notebook_run.output_root,
        "Shared H/C numbering — archive this": Path(catalogue_result["catalogue_path"]),
    }
    links = [f"- [{label}](../{path.relative_to(root).as_posix()})" for label, path in locations.items()]
    display(Markdown("## Dataset locations\n\n" + "\n".join(links)))

## Interpretation and next steps

The five figures use checksum-verified saved data from this experiment only. The first CFL baseline registered in a shared catalogue is `C000000`; read this experiment's `case_aliases.csv` for its exact baseline ID because a later frozen source version receives a new ID. Rejected proposals remain in the authoritative packets and catalogue but are not plotted as accepted EoSs. Stellar curves stop at the saved sampled peak, preserve failed-attempt gaps, and do not replace the separately refined maximum-mass result.

`dataset_40` retains final STRICT ODE tolerances but has only one stellar sampling stage, so its saved status has no per-case stellar refinement envelope. It remains experimental until matched CFL STRICT comparisons are authorized.